In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbw_gitarchive.silver.pipeline_checkpoints;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType
from delta.tables import DeltaTable

# ============================================================================
# 1. INITIALIZATION & METADATA CONFIGURATION
# ============================================================================
# We define our table names using the 3-tier Unity Catalog format: catalog.schema.table
BRONZE_TABLE = "dbw_gitarchive.bronze.raw_github_events"
TARGET_TABLE = "dbw_gitarchive.silver.github_events_cleaned"
QUARANTINE_TABLE = "dbw_gitarchive.silver.github_events_quarantine"

# In Databricks Serverless, legacy 'dbfs:/' paths are blocked. 
# We use a Unity Catalog 'Volume' path to securely save the stream's progress markers (checkpoints).
CHECKPOINT_LOCATION = "/Volumes/dbw_gitarchive/silver/pipeline_checkpoints/silver_github_events"

# We define strict, explicit schemas for the nested JSON strings in our raw data.
# This ensures that even if GitHub adds new fields tomorrow, our code only extracts what we need.
actor_schema = StructType([
    StructField("id", LongType(), True),
    StructField("login", StringType(), True),
    StructField("display_login", StringType(), True),
    StructField("url", StringType(), True),
    StructField("avatar_url", StringType(), True)
])

repo_schema = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("url", StringType(), True)
])

org_schema = StructType([
    StructField("id", LongType(), True),
    StructField("log_in", StringType(), True)
])

# ============================================================================
# 2. STREAMING READ & INLINE TRANSFORMATIONS
# ============================================================================
# 'readStream' sets up a continuous monitor on the Bronze table.
# It acts like a camera, only capturing rows that have been newly added since the last run.
streaming_bronze_df = (spark.readStream
    .format("delta")
    .table(BRONZE_TABLE)
)

# We chain our cleaning transformations together. This sets up the logical plan.
parsed_stream_df = (streaming_bronze_df
    # Add audit columns to trace data lineage (where it came from and when it was processed)
    .withColumn("audit_source_file", F.col("source_file"))
    .withColumn("audit_processed_at", F.current_timestamp())
    
    # Cast raw string datetimes into proper, highly-optimized Timestamp and Date types
    .withColumn("created_at", F.to_timestamp(F.col("created_at")))
    .withColumn("event_date", F.to_date(F.col("created_at")))
    .withColumn("event_id", F.col("id")) 
    
    # Use 'from_json' to turn raw JSON text strings into structured PySpark objects (Structs)
    .withColumn("actor_struct", F.from_json(F.col("actor"), actor_schema))
    .withColumn("repo_struct", F.from_json(F.col("repo"), repo_schema))
    .withColumn("org_struct", F.from_json(F.col("org"), org_schema))
    
    # Extract specific nested values from our new Structs using dot-notation (.) and trim whitespaces
    .withColumn("event_type", F.trim(F.col("type"))) 
    .withColumn("actor_id", F.col("actor_struct.id"))
    .withColumn("actor_username", F.trim(F.col("actor_struct.login")))
    .withColumn("repo_id", F.col("repo_struct.id"))
    .withColumn("repo_name", F.trim(F.col("repo_struct.name")))
    .withColumn("payload", F.trim(F.col("payload")))
    
    # Standardize a messy boolean column. Converts text like 'T', '1', or 'TRUE' into actual Booleans (True/False)
    .withColumn(
        "is_public",
        F.when(F.upper(F.trim(F.col("public"))).isin("T", "TRUE", "1"), True)
         .when(F.upper(F.trim(F.col("public"))).isin("F", "FALSE", "0"), False)
         .otherwise(None) # Safely flags corrupt strings as Null
    )
    
    # Handle the 'Org' column carefully. Orgs are optional on GitHub, so we format missing ones cleanly as Null
    .withColumn("org_id", F.when(F.col("org_struct.id") > 0, F.col("org_struct.id")).otherwise(None))
    .withColumn(
        "org_name", 
        F.when((F.col("org_id").isNotNull()) & (F.trim(F.col("org_struct.log_in")) != ""), F.trim(F.col("org_struct.log_in")))
         .otherwise(None)
    )
    
    # Drop intermediate columns and raw JSON blocks to keep our Silver layer tidy and compact
    .drop("id", "type", "actor", "repo", "org", "public", "actor_struct", "repo_struct", "org_struct", "source_file", "ingested_at")
)

# ============================================================================
# 3. QUALITY GATEWAY FILTERS
# ============================================================================
# We outline exactly what a "good" production record looks like.
# If a row fails any of these rules (e.g., missing ID, empty string, negative ID), it gets flagged.
validity_condition = (
    F.col("event_id").isNotNull() &
    F.col("event_type").isNotNull() & (F.col("event_type") != "") &
    F.col("actor_id").isNotNull() & (F.col("actor_id") > 0) &
    F.col("actor_username").isNotNull() & (F.col("actor_username") != "") &
    F.col("repo_id").isNotNull() & (F.col("repo_id") > 0) &
    F.col("repo_name").isNotNull() & (F.col("repo_name") != "") &
    F.col("payload").isNotNull() & (F.col("payload") != "") &
    F.col("is_public").isNotNull() &
    F.col("event_date").isNotNull()
)

# ============================================================================
# 4. MICRO-BATCH SINK FUNCTION (Serverless Compatible)
# ============================================================================
# Streaming data arrives in tiny chunks called micro-batches. 
# 'foreachBatch' pauses the stream for a brief second to process each chunk using this custom Python function.
def process_micro_batch(micro_batch_df, batch_id):
    """
    This function processes each micro-batch individually. 
    To remain compatible with Databricks Serverless compute, we do NOT use .cache() or .persist().
    """
    
    # ---------------------------------------------------------
    # SUB-STEP A: Handle Quarantine Processing
    # ---------------------------------------------------------
    # We isolate any records inside this chunk that FAIL our validity checklist
    batch_quarantine = micro_batch_df.filter(~validity_condition)
    
    # If there are any bad records, we tag them with an error reason and append them to the Quarantine table
    if batch_quarantine.count() > 0:
        (batch_quarantine
            .withColumn("quarantine_reason", F.lit("Data Quality Filter Failure"))
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(QUARANTINE_TABLE))
    
    # ---------------------------------------------------------
    # SUB-STEP B: Handle Clean Target Processing
    # ---------------------------------------------------------
    # We isolate the records inside this chunk that PASS our validity checklist
    batch_clean = micro_batch_df.filter(validity_condition)
    
    if batch_clean.count() > 0:
        # Deduplicate records that have identical event_ids *within this specific micro-batch*.
        # We sort by our audit timestamp to ensure we keep the newest record.
        window_spec = Window.partitionBy("event_id").orderBy(F.col("audit_processed_at").desc())
        deduped_batch_df = (batch_clean
            .withColumn("row_num", F.row_number().over(window_spec))
            .filter(F.col("row_num") == 1)
            .drop("row_num"))
        
        # Idempotent Upsert: Check if our target Silver table already exists
        if spark.catalog.tableExists(TARGET_TABLE):
            target_table = DeltaTable.forName(spark, TARGET_TABLE)
            
            # If the table exists, we run a MERGE.
            # If the incoming event_id already exists on that date, it updates it. If not, it inserts it.
            # This completely prevents duplicate rows if the pipeline fails and restarts!
            (target_table.alias("target")
                .merge(
                    deduped_batch_df.alias("updates"),
                    "target.event_id = updates.event_id AND target.event_date = updates.event_date"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())
        else:
            # First-Run Initialization: If the Silver table doesn't exist yet, create it fresh.
            # We partition by 'event_date' to drastically speed up downstream queries.
            (deduped_batch_df.write
                .format("delta")
                .mode("overwrite")
                .partitionBy("event_date")
                .saveAsTable(TARGET_TABLE))

# ============================================================================
# 5. START STREAM EXECUTION
# ============================================================================
# This kicks off the streaming engine configuration
query = (parsed_stream_df.writeStream
    .format("delta")
    .foreachBatch(process_micro_batch)      # Route our stream through our custom batch function above
    .option("checkpointLocation", CHECKPOINT_LOCATION) # Read/Write progress tracking markers
    
    # 'availableNow=True' acts as a hybrid. It processes all newly available records like a batch job,
    # but uses the stream's checkpoint markers to ensure it never re-processes older files.
    .trigger(availableNow=True) 
    .start()
)

# This stops the notebook cell from finishing until the streaming query completely finishes processing
query.awaitTermination()

# Test

In [0]:
%sql
-- Check if any records failed your quality gates
SELECT quarantine_reason, COUNT(*) as bad_record_count 
FROM dbw_gitarchive.silver.github_events_quarantine
GROUP BY quarantine_reason;

In [0]:
%sql
SELECT 
    event_date, 
    COUNT(*) as total_clean_records,
    COUNT(DISTINCT event_id) as unique_event_ids
FROM dbw_gitarchive.silver.github_events_cleaned
GROUP BY event_date
ORDER BY event_date DESC;

## Check for duplicates

In [0]:
%sql
SELECT 
    event_id, 
    COUNT(*) as appearance_count
FROM dbw_gitarchive.silver.github_events_cleaned
GROUP BY event_id
HAVING COUNT(*) > 1;